# NL-to-SQL Research Demo
## Comparing Prompting, Fine-tuning, and Agentic Repair on ClassicModels

**What this notebook does:**  
This is a self-contained, visual walkthrough of the dissertation research journey. No model loading or database connection is required — all examples use pre-computed outputs from the real evaluation runs.

---

### Research Question
> *Can few-shot prompting alone outperform QLoRA fine-tuning for NL-to-SQL on a closed-domain database, in a resource-constrained setting?*

### The Three Approaches Evaluated

| Approach | Description | When it matters |
|---|---|---|
| **Baseline (zero/few-shot)** | Prompt the model with 0 or 3 examples | Fast, no training cost |
| **QLoRA fine-tuning** | 4-bit quantised LoRA adapters trained on domain data | Can a tuned model beat prompting? |
| **ReAct loop** | Generate → validate → repair using execution feedback | Can iterative self-correction add value? |

### Models
- **Llama-3-8B-Instruct** (Meta)  
- **Qwen-2.5-7B-Instruct** (Alibaba)  

Both run in 4-bit NF4 quantisation on a single Colab T4 GPU (~16 GB VRAM).

In [ ]:
# No GPU, model, or database needed — all outputs are pre-computed
print("Demo ready. No GPU required.")

---
## 1. The Database: ClassicModels

All experiments run against a single fixed MySQL database — **ClassicModels** — a standard retail dataset with 8 tables covering customers, orders, products, and payments.  
The benchmark contains **200 hand-crafted NL questions** paired with gold SQL answers.

In [ ]:
# ClassicModels schema — 8 tables used in all experiments
tables = {
    "customers":    ["customerNumber (PK)", "customerName", "country", "creditLimit"],
    "orders":       ["orderNumber (PK)", "customerNumber (FK)", "status", "orderDate"],
    "orderdetails": ["orderNumber (FK)", "productCode (FK)", "quantityOrdered", "priceEach"],
    "products":     ["productCode (PK)", "productName", "productLine (FK)", "MSRP"],
    "productlines": ["productLine (PK)", "textDescription"],
    "payments":     ["customerNumber (FK)", "checkNumber", "amount", "paymentDate"],
    "employees":    ["employeeNumber (PK)", "reportsTo (FK)", "jobTitle", "officeCode (FK)"],
    "offices":      ["officeCode (PK)", "city", "country", "phone"],
}

print("ClassicModels — 8 tables")
print("=" * 50)
for table, cols in tables.items():
    print(f"\n  {table}")
    for col in cols:
        print(f"    - {col}")

print("\nKey foreign keys:")
fks = [
    "orders.customerNumber        -> customers.customerNumber",
    "orderdetails.orderNumber     -> orders.orderNumber",
    "orderdetails.productCode     -> products.productCode",
    "products.productLine         -> productlines.productLine",
    "payments.customerNumber      -> customers.customerNumber",
    "employees.officeCode         -> offices.officeCode",
]
print()
for fk in fks:
    print(f"  {fk}")

---
## 2. Evaluation Metrics

Four metrics are used, each measuring a different level of correctness:

| Metric | Full name | What it tests | Score range |
|--------|-----------|---------------|-------------|
| **VA** | Valid SQL | Does the SQL parse and execute without error? | 0–1 |
| **EM** | Exact Match | Does the predicted SQL match the gold string after normalisation? | 0–1 |
| **EX** | Execution Accuracy | Do the predicted and gold queries return the same result set? | 0–1 |
| **TS** | Test-Suite Accuracy | Does EX hold across 10 database replicas with different values? | 0–1 |

> **Why EX > EM?**  Multiple valid SQL queries can answer the same question (`COUNT(*)` vs `COUNT(col)`). EX captures this; EM does not.

In [ ]:
# Show what each metric actually measures on a single example
NLQ  = "How many customers are in France?"
GOLD = "SELECT COUNT(*) FROM customers WHERE country = 'France';"

preds = [
    (GOLD,                                                                    True,  True,  True,  True,  "perfect match"),
    ("SELECT COUNT(customerNumber) FROM customers WHERE country = 'France';", True,  False, True,  True,  "same result, different syntax"),
    ("SELECT * FROM customers WHERE country = 'France';",                     True,  False, False, False, "wrong projection (SELECT *)"),
    ("SELECT COUNT(*) FROM orders WHERE country = 'France';",                 False, False, False, False, "schema error (orders has no country)"),
]

print(f"NLQ:  {NLQ}")
print(f"Gold: {GOLD}")
print()
print(f"  {'VA':>3}  {'EM':>3}  {'EX':>3}  {'TS':>3}    notes")
print(f"  {'--':>3}  {'--':>3}  {'--':>3}  {'--':>3}    -----")
for sql, va, em, ex, ts, note in preds:
    check = lambda b: "✓" if b else "✗"
    print(f"  {check(va):>3}  {check(em):>3}  {check(ex):>3}  {check(ts):>3}    {note}")
    print(f"       {sql}")
    print()

print("VA = did the SQL execute without error?")
print("EM = does it exactly match the gold string?")
print("EX = does it return the same result set?")
print("TS = does EX hold across 10 database replicas?")

---
## 3. Approach 1 — Baseline Prompting (Zero-shot vs. Few-shot)

The **baseline** passes a system prompt + schema + (optionally) worked examples directly into the model.  
No training, no fine-tuning — just carefully structured text.

### k=0 (zero-shot) vs. k=3 (3-shot)
- **k=0**: The model sees the schema and the question only
- **k=3**: The model also sees 3 NLQ→SQL examples drawn from the benchmark pool

In [ ]:
# Show what the prompt looks like for k=0 vs k=3

SCHEMA = (
    "customers(customerNumber, customerName, country, creditLimit)\n"
    "orders(orderNumber, customerNumber, status, orderDate)\n"
    "orderdetails(orderNumber, productCode, quantityOrdered, priceEach)\n"
    "products(productCode, productName, productLine, MSRP)\n"
    "payments(customerNumber, checkNumber, amount)\n"
    "Join hints: customers.customerNumber = orders.customerNumber; ..."
)

EXEMPLARS = [
    ("List all customer names in Spain.",
     "SELECT customerName FROM customers WHERE country = 'Spain';"),
    ("How many products are in each product line?",
     "SELECT productLine, COUNT(*) FROM products GROUP BY productLine;"),
    ("What is the total payment amount received?",
     "SELECT SUM(amount) FROM payments;"),
]

TARGET = "Top 3 countries by number of orders."


def print_prompt(k, schema, exemplars, target):
    print(f"--- PROMPT  k={k} {'(zero-shot)' if k == 0 else '(few-shot)'} ---")
    print("[SYSTEM]")
    print("  You are a MySQL analyst. Return one SQL SELECT statement.")
    print()
    print("[SCHEMA]")
    for line in schema.strip().split("\n"):
        print(f"  {line}")
    print()
    if k > 0:
        print(f"[EXAMPLES]  ({k} worked examples)")
        for i, (nlq, sql) in enumerate(exemplars[:k], 1):
            print(f"  {i}. Q: {nlq}")
            print(f"     A: {sql}")
        print()
    print(f"[USER]  Convert to SQL:")
    print(f"  {target}")
    print()


print_prompt(0, SCHEMA, EXEMPLARS, TARGET)
print_prompt(3, SCHEMA, EXEMPLARS, TARGET)

---
## 4. Approach 2 — QLoRA Fine-tuning

**QLoRA** (Dettmers et al., 2023) fine-tunes a quantised model by adding small trainable rank-decomposition matrices (LoRA adapters) to the attention layers. The base model weights stay frozen in 4-bit NF4 format.

- **Training data**: 160 items from the 200-item benchmark (held-out 40 items not used)
- **Hardware**: single T4 GPU (~16 GB VRAM), ~25 minutes per run  
- **Key question**: does domain-specific fine-tuning outperform few-shot prompting with zero training?

In [ ]:
# QLoRA architecture — text overview
print("QLoRA Architecture")
print("=" * 50)
print()
print("  [Training data]  160 NLQ->SQL pairs")
print("         |")
print("         v")
print("  +---------------------------+   +-----------------------+")
print("  |  Frozen Base Model        |   |  LoRA Adapters        |")
print("  |  4-bit NF4, ~4 GB VRAM   | <-|  rank=32, alpha=64    |")
print("  |  weights FROZEN           |   |  ~6M trainable params |")
print("  +---------------------------+   +-----------------------+")
print("                 |")
print("                 v  (adapters injected into attention layers)")
print("  +---------------------------+")
print("  |  QLoRA Eval Model         |")
print("  |  base + adapters merged   |")
print("  +---------------------------+")
print()
print("Key numbers:")
print("  Base model:     ~4 GB VRAM (4-bit NF4 quantised)")
print("  Adapter params: ~6M  (<1% of base model)")
print("  Training time:  ~25 min on T4 GPU")
print("  Training data:  160 items  (40 held out for test)")

---
## 5. Approach 3 — ReAct Loop (Generate → Validate → Repair)

The **ReAct** (Yao et al., 2023) pattern interleaves *reasoning* and *acting* in a loop.  
For NL-to-SQL this means:

1. **Get schema** — load table/column definitions  
2. **Generate SQL** — few-shot prompted generation  
3. **Validate SQL** — static schema checks (unknown columns, SELECT \*, ambiguity)  
4. **Validate constraints** — NLQ-derived semantic checks (required columns, aggregations, ORDER BY)  
5. **Execute SQL** — actually run against the database  
6. **Repair** — if any check fails, feed the error back to the model and regenerate  

Steps 3–6 repeat up to `max_repairs` times.

In [ ]:
# ReAct loop — ASCII step diagram
print("ReAct Loop: Generate -> Validate -> Execute -> Repair")
print("=" * 55)
print()
print("  Step 1:  get_schema")
print("             load table/column definitions")
print("             |")
print("  Step 2:  generate_sql   (k=3 few-shot prompt)")
print("             |")
print("  Step 3:  validate_sql   (static schema check)")
print("             |-- FAIL --> repair_sql --> back to Step 2")
print("             |")
print("  Step 4:  validate_constraints   (NLQ semantic check)")
print("             |-- FAIL --> repair_sql --> back to Step 2")
print("             |")
print("  Step 5:  run_sql   (execute against the database)")
print("             |-- ERROR --> repair_sql --> back to Step 2")
print("             |")
print("  Step 6:  STOP  -- return final SQL")
print()
print("  Budget: max_repairs = 3  (if exhausted, return best SQL seen)")

---
## 6. ReAct Loop — Live Trace Walkthrough

The two traces below come directly from the real evaluation run (`results/agent/results_react_200.json`).  
Both show a case where the first SQL generated was **invalid** and the loop successfully **repaired** it.

In [ ]:
# Real trace from results/agent/results_react_200.json
# NLQ: "Top 3 countries by number of orders."

def print_trace(trace, nlq, gold):
    print(f"NLQ:  {nlq}")
    print(f"Gold: {gold}")
    print()
    for t in trace:
        tag = f"  [{t['tag'].upper()}]" if t.get('tag') else ""
        print(f"  [{t['step']}] {t['action']}{tag}")
        for line in t['obs'].split('\n'):
            print(f"        {line}")
        if t.get('hint'):
            print(f"        hint: {t['hint']}")
        print()


trace_1 = [
    {'step': 1, 'action': 'get_schema',
     'obs': 'Loaded 8 tables: customers, orders, orderdetails, products, ...'},
    {'step': 2, 'action': 'generate_sql', 'tag': 'generated',
     'obs': "SELECT country, COUNT(orderNumber) AS order_count\n"
            "FROM orders\nGROUP BY country ORDER BY order_count DESC LIMIT 3;"},
    {'step': 3, 'action': 'validate_sql', 'tag': 'fail',
     'obs': "FAIL -- unknown_column: orders.country\n"
            "(column 'country' does not exist in orders; it is in customers)"},
    {'step': 4, 'action': 'repair_sql', 'tag': 'repaired',
     'obs': "SELECT c.country, COUNT(o.orderNumber) AS order_count\n"
            "FROM customers c\n"
            "JOIN orders o ON c.customerNumber = o.customerNumber\n"
            "GROUP BY c.country ORDER BY order_count DESC LIMIT 3;",
     'hint': "unknown_column: country -- schema shows customers.country"},
    {'step': 5, 'action': 'validate_sql',         'tag': 'pass', 'obs': 'OK -- schema_ok'},
    {'step': 6, 'action': 'validate_constraints',  'tag': 'pass', 'obs': 'OK'},
    {'step': 7, 'action': 'run_sql',               'tag': 'pass', 'obs': 'SUCCESS -- 3 rows returned'},
    {'step': 7, 'action': 'STOP',                  'tag': 'stop', 'obs': 'reason: success'},
]

print("--- Trace 1: missing JOIN detected and repaired ---")
print()
print_trace(
    trace_1,
    nlq="Top 3 countries by number of orders.",
    gold="SELECT c.country, COUNT(*) FROM customers c JOIN orders o ON c.customerNumber = o.customerNumber GROUP BY c.country ORDER BY COUNT(*) DESC LIMIT 3;",
)

In [ ]:
# Real trace from results/agent/results_react_200.json
# NLQ: "How many orders from customers in Australia have status 'Disputed'?"

trace_2 = [
    {'step': 1, 'action': 'get_schema',
     'obs': 'Loaded 8 tables: customers, orders, orderdetails, products, ...'},
    {'step': 2, 'action': 'generate_sql', 'tag': 'generated',
     'obs': "SELECT COUNT(*) AS disputedOrders FROM orders\n"
            "WHERE status = 'Disputed'\n"
            "AND customerNumber IN (SELECT customerNumber FROM customers WHERE country = 'Australia');"},
    {'step': 3, 'action': 'validate_sql', 'tag': 'fail',
     'obs': "FAIL -- ambiguous_column: customerNumber\n"
            "(exists in both 'orders' and 'customers' -- needs table qualifier)"},
    {'step': 4, 'action': 'repair_sql', 'tag': 'repaired',
     'obs': "SELECT COUNT(*) AS disputedOrders\n"
            "FROM orders o\n"
            "JOIN customers c ON o.customerNumber = c.customerNumber\n"
            "WHERE o.status = 'Disputed' AND c.country = 'Australia';",
     'hint': "ambiguous_column: customerNumber -- qualify with table alias"},
    {'step': 5, 'action': 'validate_sql',         'tag': 'pass', 'obs': 'OK -- schema_ok'},
    {'step': 6, 'action': 'validate_constraints',  'tag': 'pass', 'obs': 'OK'},
    {'step': 7, 'action': 'run_sql',               'tag': 'pass', 'obs': 'SUCCESS -- 1 row returned'},
    {'step': 7, 'action': 'STOP',                  'tag': 'stop', 'obs': 'reason: success'},
]

print("--- Trace 2: ambiguous column qualified with table alias ---")
print()
print_trace(
    trace_2,
    nlq="How many orders from customers in Australia have status 'Disputed'?",
    gold="SELECT COUNT(*) FROM orders o JOIN customers c ON o.customerNumber = c.customerNumber WHERE c.country = 'Australia' AND o.status = 'Disputed';",
)

---
## 7. Results — All Conditions Compared

The table below summarises results from the full evaluation across 8 conditions.  
Numbers are **means across seeds** (3 seeds × 200 items = 600 pairs per condition, except Qwen Base k=0 which has 2 seeds).

In [ ]:
# Results summary — all 8 conditions (means across seeds)
# Source: results/analysis/stats_mean_median_shapiro.csv

conditions = [
    ("Llama Base  k=0", 0.765, 0.005, 0.410, None),
    ("Llama Base  k=3", 0.870, 0.298, 0.517, 0.559),
    ("Llama QLoRA k=0", 0.815, 0.000, 0.440, None),
    ("Llama QLoRA k=3", 0.855, 0.285, 0.465, None),
    ("Qwen  Base  k=0", 0.805, 0.010, 0.480, None),
    ("Qwen  Base  k=3", 0.892, 0.357, 0.610, 0.667),
    ("Qwen  QLoRA k=0", 0.805, 0.010, 0.435, None),
    ("Qwen  QLoRA k=3", 0.895, 0.302, 0.530, 0.566),
]

print("Evaluation Results — All 8 Conditions  (mean across seeds)")
print("=" * 65)
print(f"  {'Condition':<20}  {'VA':>6}  {'EM':>6}  {'EX':>6}  {'TS':>6}")
print(f"  {'-'*20}  {'------':>6}  {'------':>6}  {'------':>6}  {'------':>6}")

for label, va, em, ex, ts in conditions:
    ts_str = f"{ts:.3f}" if ts is not None else "  N/A"
    marker = "  <-- k=3" if "k=3" in label else ""
    print(f"  {label:<20}  {va:>6.3f}  {em:>6.3f}  {ex:>6.3f}  {ts_str:>6}{marker}")

print()
print("Notes:")
print("  - Qwen Base k=0 has 2 seeds (seed=27 still missing); all others have 3 seeds")
print("  - Llama QLoRA k=3 has no TS score (ts_enabled=False in that run)")
print("  - TS = Test-Suite Accuracy; N/A = not evaluated")

In [ ]:
# Key finding: Base vs QLoRA at k=3 — EX metric (the primary metric)

print("Base vs QLoRA at k=3 — Execution Accuracy (EX)")
print("=" * 55)
print()
print("  Llama-3-8B-Instruct")
print(f"    k=0:  Base={0.410:.3f}   QLoRA={0.440:.3f}")
print(f"    k=3:  Base={0.517:.3f}   QLoRA={0.465:.3f}")
delta_llama = (0.517 - 0.465) * 100
print(f"    --> Base beats QLoRA at k=3 by +{delta_llama:.1f}pp EX")
print()
print("  Qwen-2.5-7B-Instruct")
print(f"    k=0:  Base={0.480:.3f}   QLoRA={0.435:.3f}")
print(f"    k=3:  Base={0.610:.3f}   QLoRA={0.530:.3f}")
delta_qwen = (0.610 - 0.530) * 100
print(f"    --> Base beats QLoRA at k=3 by +{delta_qwen:.1f}pp EX")
print()
print("-" * 55)
print("Core finding: Base (prompting-only) consistently outperforms")
print("QLoRA fine-tuning at k=3 on both model families.")
print()
print("Interpretation: In a resource-constrained setting (single GPU,")
print("small training set), domain fine-tuning does not compensate")
print("for the reduction in few-shot generalisation capacity.")

---
## 8. Statistical Significance — Paired t-tests

All differences are tested with **paired t-tests** matched on `(seed, example_id)`.  
Because each metric is a binary 0/1 variable, the distributions are non-normal (Shapiro-Wilk rejects in every case) — but at n=400–600 pairs, the Central Limit Theorem justifies the t-test.

H₀: *mean(condition_A) = mean(condition_B)*  
Significance threshold: α = 0.05

In [ ]:
# Statistical significance — paired t-tests (alpha = 0.05)
# Source: results/analysis/stats_paired_ttests.csv

results_stats = [
    # (comparison,              metric, left_mean, right_mean, p_value,  decision)
    ('Llama Base k0->k3',      'EX',   0.410,     0.517,      1.10e-11, 'reject H0'),
    ('Llama Base k0->k3',      'VA',   0.765,     0.870,      1.81e-11, 'reject H0'),
    ('Llama QLoRA k0->k3',     'EX',   0.440,     0.465,      0.0547,   'FAIL to reject H0'),  # key!
    ('Llama QLoRA k0->k3',     'VA',   0.815,     0.855,      0.0113,   'reject H0'),
    ('Qwen Base k0->k3',       'EX',   0.480,     0.613,      1.50e-07, 'reject H0'),
    ('Qwen Base k0->k3',       'VA',   0.805,     0.890,      2.84e-06, 'reject H0'),
    ('Qwen QLoRA k0->k3',      'EX',   0.435,     0.530,      1.37e-09, 'reject H0'),
    ('Llama Base->QLoRA @k3',  'EX',   0.517,     0.465,      7.40e-04, 'reject H0  [*]'),
    ('Qwen Base->QLoRA @k3',   'EX',   0.610,     0.530,      8.60e-06, 'reject H0  [*]'),
    ('Qwen Base->QLoRA @k3',   'TS',   0.675,     0.578,      9.29e-07, 'reject H0  [*]'),
]

print("Paired t-test Results  (alpha = 0.05, matched by seed + example_id)")
print("=" * 80)
print(f"  {'Comparison':<26}  {'M':<3}  {'Left':>6}  {'Right':>6}  {'Delta':>7}  {'p-value':>10}  Decision")
print(f"  {'-'*26}  {'--':<3}  {'------':>6}  {'------':>6}  {'-------':>7}  {'----------':>10}  --------")

for comp, metric, lm, rm, pval, decision in results_stats:
    delta = (rm - lm) * 100
    p_str = f"{pval:.2e}" if pval < 0.001 else f"{pval:.4f}"
    flag = "  <-- not sig!" if "FAIL" in decision else ("  <-- core claim" if "[*]" in decision else "")
    print(f"  {comp:<26}  {metric:<3}  {lm:>6.3f}  {rm:>6.3f}  {delta:>+6.1f}pp  {p_str:>10}  {decision}{flag}")

print()
print("[*] = core dissertation claim: Base beats QLoRA at k=3, statistically significant")
print()
print("Key result:")
print("  Llama QLoRA k0->k3 EX uplift is NOT significant (p=0.055 > 0.05).")
print("  Fine-tuning appears to suppress the model's ability to use few-shot examples.")

---
## 9. ReAct Results (Preliminary — Single Seed)

The ReAct loop (Llama + QLoRA adapter, k=3, seed=7, n=200) achieves:

| Metric | Baseline k=3 | ReAct | Δ |
|--------|-------------|-------|---|
| VA     | 0.870       | **0.915** | +4.5pp |
| EM     | 0.298       | **0.480** | +18.2pp |
| EX     | 0.517       | **0.605** | +8.8pp |
| TS     | 0.559       | **0.639** | +8.0pp |

> ⚠️ **Single-seed result only** — seeds 17 and 27 needed before paired significance tests are valid.

In [ ]:
# ReAct vs Baseline k=3 — Llama-3-8B (preliminary, single seed=7)
# Source: results/agent/results_react_200.json

metrics      = ['VA',   'EM',   'EX',   'TS']
baseline_k3  = [0.870,  0.298,  0.517,  0.559]
react_vals   = [0.915,  0.480,  0.605,  0.639]

print("ReAct vs Baseline k=3 — Llama-3-8B  (preliminary, single seed=7, n=200)")
print("=" * 65)
print(f"  {'Metric':<8}  {'Baseline k=3':>14}  {'ReAct':>8}  {'Delta':>8}")
print(f"  {'-'*8}  {'------------':>14}  {'-----':>8}  {'-----':>8}")

for m, bv, rv in zip(metrics, baseline_k3, react_vals):
    delta = (rv - bv) * 100
    print(f"  {m:<8}  {bv:>14.3f}  {rv:>8.3f}  {delta:>+7.1f}pp")

print()
print("ReAct conditions: Llama-3-8B-Instruct + QLoRA adapter, k=3, max_repairs=3")
print()
print("Interpretation:")
print("  ReAct gains +8.8pp EX over Baseline k=3 through execution-guided repair.")
print("  EM gain (+18.2pp) is especially large — the repair loop corrects structural")
print("  SQL errors that reduce exact-match even when results are correct.")
print()
print("WARNING: Single-seed result only.")
print("  Seeds 17 and 27 needed before paired significance tests are valid.")

---
## 10. Key Findings

### Finding 1 — Few-shot prompting beats fine-tuning (statistically significant)

At k=3, the base (no fine-tuning) model outperforms QLoRA on **execution accuracy** for both model families:

- **Llama**: Base 51.7% vs QLoRA 46.5% — Δ = −5.2pp (p = 0.00074) ✓
- **Qwen**: Base 61.0% vs QLoRA 53.0% — Δ = −8.0pp (p = 8.6 × 10⁻⁶) ✓

This suggests that in a *constrained resource setting* (single GPU, small training set), **domain adaptation via QLoRA does not compensate for the loss of few-shot generalisation capacity**.

### Finding 2 — Llama QLoRA shows attenuated ICL benefit

While Llama Base gains +10.7pp EX from k=0→k=3 (p < 10⁻¹¹), **Llama QLoRA only gains +2.5pp and this is NOT statistically significant** (p = 0.055).  
This implies fine-tuning partially suppresses the model's ability to leverage in-context examples.

### Finding 3 — ReAct adds further gains on top of baseline

The execution-guided repair loop (preliminary, single seed) gains +8.8pp EX over the Llama Base k=3 baseline.  
This positions ReAct as a practical extension layer rather than a replacement for prompting.

---

### Research Journey Summary

```
Zero-shot baseline  →  Few-shot (k=3)  →  QLoRA fine-tuning  →  ReAct loop
    EX ≈ 41–48%        EX ≈ 52–61%        EX ≈ 47–53%          EX ≈ 61%*

        ↑ prompting wins over fine-tuning ↑          ↑ repair adds more ↑
                   (* preliminary, Llama only)
```